# Reconia — RoBERTa + bố cục 2D trên SROIE / CORD
Cùng dữ liệu/split với GNN đã kiểm chứng; chạy smoke trước full training. Mỗi từ OCR được xử lý qua cửa sổ 512 token.
SROIE xuất thêm chỉ số bốn key gốc, tách biệt khỏi BIO và giá trị đoạn OCR.


In [ ]:
DATASET = "sroie"  # "sroie" or "cord"
RUN_MODE = "smoke"
RUN_ID = f"roberta-{DATASET}-seed42-receipts-v1"
EPOCHS = 20
SEED = 42
RESUME_FROM = ""
BACKBONE_REPO = "FacebookAI/roberta-base"
BACKBONE_REVISION = "e2da8e2f811d1448a5b465c236feacd80ffbac7b"


In [ ]:
import os, sys, json, hashlib, shutil, subprocess, zipfile, re
from pathlib import Path
assert DATASET in {"sroie", "cord"}
assert RUN_MODE in {"smoke", "train"}
assert re.fullmatch(r"[a-zA-Z0-9_-]+", RUN_ID), "Use a simple run ID"
assert EPOCHS > 0
assert not RESUME_FROM or RUN_MODE == "train", "Resume is only supported for full training"
# Keep Kaggle's existing CUDA PyTorch, NumPy and pandas installations.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "python-dotenv==1.0.1", "transformers==4.57.6", "datasets==5.0.1", "pyarrow==23.0.1", "Pillow>=10.0"], check=True)
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
CODE = Path("/kaggle/temp/reconia-code") / RUN_ID
DATA = Path("/kaggle/temp/reconia-receipts") / RUN_ID
bundle_manifests = list(INPUT.rglob("bundle-manifest.json"))
assert len(bundle_manifests) == 1, "Attach exactly one receipt bundle manifest"
bundle_manifest = json.loads(bundle_manifests[0].read_text())
def extract_bundle(filename, destination):
    matches = list(INPUT.rglob(filename))
    assert len(matches) == 1, f"Attach exactly one Reconia bundle: expected {filename}, found {matches}"
    expected_archive = bundle_manifest[filename]
    assert matches[0].stat().st_size == expected_archive["bytes"], filename
    with matches[0].open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == expected_archive["sha256"], filename
    if destination.exists() and any(destination.iterdir()):
        raise RuntimeError("Extraction directory already used. Restart the session or choose a new RUN_ID.")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0]) as z:
        assert len(z.namelist()) == len(set(z.namelist())), "Duplicate archive paths"
        for name in z.namelist():
            assert (destination / name).resolve().is_relative_to(destination.resolve())
        z.extractall(destination)
extract_bundle("reconia-code.bin", CODE)
provenance = json.loads((CODE / "source-provenance.json").read_text())
for name, digest in provenance["files_sha256"].items():
    assert hashlib.sha256((CODE / name).read_bytes()).hexdigest() == digest, name
if "receipt-source-reference.json" in bundle_manifest:
    def verified_input(filename):
        matches = list(INPUT.rglob(filename))
        assert len(matches) == 1, filename
        path = matches[0]
        assert path.stat().st_size == bundle_manifest[filename]["bytes"], filename
        assert hashlib.sha256(path.read_bytes()).hexdigest() == bundle_manifest[filename]["sha256"], filename
        return path
    reference = verified_input("receipt-source-reference.json")
    targets = verified_input(f"{DATASET}-field-value-targets.json")
    subprocess.run([sys.executable, "-u", "-m", "scripts.fetch_kaggle_receipts",
                    "--dataset", DATASET, "--destination", str(DATA),
                    "--reference", str(reference), "--targets", str(targets)], check=True, cwd=CODE)
else:
    extract_bundle(f"{DATASET}-receipts.bin", DATA)
manifest = json.loads((DATA / "checksums.json").read_text())
for name, digest in manifest.items():
    path = (DATA / name).resolve()
    assert path.is_relative_to(DATA.resolve())
    with path.open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == digest, name
provenance = json.loads((CODE / "source-provenance.json").read_text())
for name, digest in provenance["files_sha256"].items():
    assert hashlib.sha256((CODE / name).read_bytes()).hexdigest() == digest, name
os.chdir(CODE)
sys.path.insert(0, str(CODE))
os.environ[f"{DATASET.upper()}_DIR"] = str(DATA / DATASET)
# A stale OCR-export override would otherwise silently bypass these verified images/labels.
os.environ.pop(f"{DATASET.upper()}_EXPORT_DIR", None)
if DATASET == "sroie":
    split_bytes = (DATA / "sroie/splits.json").read_bytes()
    mapping = json.loads((DATA / "sroie/source-id-map.json").read_text())
    assert hashlib.sha256(split_bytes).hexdigest() == mapping["split_sha256"]
    assert json.loads(split_bytes) == json.loads((CODE / "app/data/sroie_splits.json").read_text())
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = str(SEED)
import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator in Notebook Settings"
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
from app.data import SroieAdapter, CordAdapter
adapter = SroieAdapter() if DATASET == "sroie" else CordAdapter()
counts = {s: len(adapter.document_ids(s)) for s in ("train", "val", "test")}
expected = {"sroie": {"train": 500, "val": 63, "test": 63}, "cord": {"train": 800, "val": 100, "test": 100}}
assert counts == expected[DATASET], counts
print("Verified dataset:", counts)
del adapter

from huggingface_hub import snapshot_download
MODEL = snapshot_download(repo_id=BACKBONE_REPO, revision=BACKBONE_REVISION,
                          cache_dir="/kaggle/temp/huggingface",
                          allow_patterns=["config.json", "model.safetensors", "tokenizer.json",
                                          "tokenizer_config.json", "vocab.json", "merges.txt"])
os.environ["ROBERTA_MODEL_PATH"] = MODEL
print("MODE:", RUN_MODE.upper(), "| Backbone:", BACKBONE_REPO, BACKBONE_REVISION)


In [ ]:
from datetime import datetime, timezone
OUT = WORK / "reconia-runs" / (RUN_ID + ("-smoke" if RUN_MODE == "smoke" else ""))
CHECKPOINTS = OUT / "checkpoints"
REPORTS = OUT / "reports"
CACHE = Path("/kaggle/temp/reconia-graphs") / OUT.name
if any(CHECKPOINTS.glob("*.pt")):
    raise RuntimeError("This RUN_ID already has checkpoints. Choose a new RUN_ID; use RESUME_FROM to continue.")
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
resume_args = []
if RESUME_FROM:
    previous = Path(RESUME_FROM)
    previous_config = json.loads((previous.parent / "run-config.json").read_text())
    assert previous_config["mode"] == "train", "Cannot resume a smoke run as a full experiment"
    assert previous_config["dataset"] == DATASET and previous_config["seed"] == SEED
    assert previous_config["epochs"] == EPOCHS, "Keep the original total training schedule"
    assert previous_config["dataset_checksums_sha256"] == hashlib.sha256((DATA / "checksums.json").read_bytes()).hexdigest(), "Resume data differs"
    assert previous_config["source"]["files_sha256"] == provenance["files_sha256"], "Resume code differs"
    for filename in (f"roberta_{DATASET}_latest.pt", f"roberta_{DATASET}_best.pt"):
        assert (previous / filename).is_file(), f"Missing resume artifact: {filename}"
        shutil.copy2(previous / filename, CHECKPOINTS / filename)
    assert previous_config["backbone_repo"] == BACKBONE_REPO
    assert previous_config["backbone_revision"] == BACKBONE_REVISION
    from app.data.roberta_dataset import WINDOW_POLICY
    saved = torch.load(CHECKPOINTS / f"roberta_{DATASET}_latest.pt", map_location="cpu", weights_only=False)
    assert saved.get("window_policy") == WINDOW_POLICY
    assert saved["epoch"] < EPOCHS, "Already finished"
    del saved
    resume_args = ["--resume", str(CHECKPOINTS / f"roberta_{DATASET}_latest.pt")]
command = [sys.executable, "-u", "-m", "app.training.train_roberta",
           "--dataset", DATASET, "--device", "cuda", "--seed", str(SEED),
           "--epochs", str(EPOCHS if RUN_MODE == "train" else 1),
           "--lr", "0.00005", "--head-lr", "0.0002", "--batch-size", "4",
           "--batch-accum", "4", "--patience", "5", "--weighted-loss", "true",
           "--cache-dir", str(CACHE), "--save-dir", str(CHECKPOINTS), "--report-dir", str(REPORTS)]
if RUN_MODE == "smoke":
    command += ["--limit-train", "8", "--limit-val", "4"]
command += resume_args
configuration = {"mode": RUN_MODE, "run_id": RUN_ID, "epochs": EPOCHS, "seed": SEED,
                 "resume_from": RESUME_FROM, "counts": counts, "command": command,
                 "gpu": torch.cuda.get_device_name(0), "started_utc": datetime.now(timezone.utc).isoformat(),
                 "source": provenance, "dataset": DATASET, "dataset_checksums_sha256": hashlib.sha256((DATA / "checksums.json").read_bytes()).hexdigest()}
configuration.update(backbone_repo=BACKBONE_REPO, backbone_revision=BACKBONE_REVISION,
                     window_policy="all-windows-first-subword-v1")
if "receipt-source-reference.json" in bundle_manifest:
    configuration["receipt_source_reference_sha256"] = bundle_manifest["receipt-source-reference.json"]["sha256"]
    configuration["data_transport"] = "pinned-public-download-with-reviewed-content-hashes"
(OUT / "run-config.json").write_text(json.dumps(configuration, indent=2))
(OUT / "requirements-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print("Output:", OUT)
print("MODE:", RUN_MODE.upper(), "| Training documents:", counts["train"] if RUN_MODE == "train" else 8)
print("Launching:", " ".join(command))
# Train in a child process so stopping the notebook does not hide Python errors.
subprocess.run(command, check=True, cwd=CODE)


In [ ]:
if RUN_MODE == "smoke":
    # Reload weights in a fresh process; use validation, never test for debugging.
    subprocess.run([sys.executable, "-u", "-m", "app.evaluation.predict",
                    "--model", "roberta", "--dataset", DATASET, "--split", "val",
                    "--checkpoint", str(CHECKPOINTS / f"roberta_{DATASET}_best.pt"),
                    "--limit", "4", "--device", "cuda", "--report-dir", str(REPORTS)], check=True, cwd=CODE)
    print("SMOKE PASSED. For the full experiment set RUN_MODE = 'train' and Save & Run All.")
else:
    evidence = REPORTS / f"predictions/predictions_roberta_{DATASET}_test.metrics.json"
    metrics = json.loads(evidence.read_text())
    assert metrics["complete"] and metrics["num_documents"] == metrics["expected_num_documents"] == counts["test"]
    print("Full local test verified:", {k: metrics[k] for k in ("precision", "recall", "f1", "protocol")})
    print("OCR-derived field-value F1:", metrics["value_metrics"]["f1"])
if DATASET == "sroie":
    score_split = "val" if RUN_MODE == "smoke" else "test"
    suffix = "_sample4" if RUN_MODE == "smoke" else ""
    score_command = [sys.executable, "-u", "-m", "app.evaluation.sroie_values",
                     "--predictions", str(REPORTS / f"predictions/predictions_roberta_sroie_{score_split}{suffix}.json"),
                     "--targets", str(DATA / "sroie/field-value-targets.json"), "--split", score_split,
                     "--output", str(REPORTS / f"predictions/roberta_sroie_{score_split}{suffix}.four-field.metrics.json")]
    if RUN_MODE == "smoke":
        score_command += ["--limit", "4"]
    subprocess.run(score_command, check=True, cwd=CODE)
archive = shutil.make_archive(str(WORK / OUT.name), "zip", root_dir=OUT.parent, base_dir=OUT.name)
from IPython.display import FileLink, display
print("Download and retain both best/latest checkpoints, reports and run configuration.")
os.chdir(WORK)
display(FileLink(str(Path(archive).relative_to(WORK))))
